In [31]:
import os
import re
import pandas as pd

from joblib import Parallel, delayed

from langdetect import (
    detect_langs,
    LangDetectException,
    DetectorFactory
)

In [32]:
DetectorFactory.seed = 42

In [33]:
column_names = [
    "comment_id",
    "entry_id",
    "user",
    "source",
    "url",
    "extra1",
    "extra2",
    "timestamp",
    "comment"
]

In [34]:
def clean_text(text):

    if pd.isna(text):
        return None

    text = str(text)

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+",
        "",
        text
    )

    # Remove HTML tags
    text = re.sub(
        r"<.*?>",
        "",
        text
    )

    # Remove extra whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text

In [35]:
def is_english(text, min_prob=0.80):

    if pd.isna(text):
        return False

    text = str(text).strip()

    if text == "":
        return False

    try:

        languages = detect_langs(
            text[:500]
        )

        return any(
            lang.lang == "en"
            and lang.prob >= min_prob
            for lang in languages
        )

    except LangDetectException:
        return False

In [36]:
def detect_english_parallel(
    texts,
    n_jobs=4
):

    return Parallel(
        n_jobs=n_jobs
    )(
        delayed(is_english)(text)
        for text in texts
    )

In [37]:
raw_file = (
    "../data/raw/comments_extracted/"
    "commentAugSept.csv"
)

output_file = (
    "../data/processed/"
    "clean_comments.csv"
)

os.makedirs(
    "../data/processed",
    exist_ok=True
)

In [40]:
chunk_size = 10_000
max_chunks = 40

total_rows = 0
total_kept = 0

first_chunk = True

In [41]:
if os.path.exists(output_file):
    os.remove(output_file)

In [42]:
for chunk_number, chunk in enumerate(

    pd.read_csv(
        raw_file,
        sep="\t",
        header=None,
        names=column_names,
        chunksize=chunk_size,
        dtype=str
    ),

    start=1
):

    # Stop after 40 chunks
    if chunk_number > max_chunks:
        break

    total_rows += len(chunk)

    # Replace \N with missing value
    chunk = chunk.replace(
        "\\N",
        pd.NA
    )

    # Remove unnecessary columns
    chunk = chunk.drop(
        columns=[
            "extra1",
            "extra2"
        ]
    )

    # Clean comment text
    chunk["comment_clean"] = (
        chunk["comment"]
        .apply(clean_text)
    )

    # Remove missing comments
    chunk = chunk.dropna(
        subset=["comment_clean"]
    )

    # Remove empty comments
    chunk = chunk[
        chunk["comment_clean"]
        .str.strip()
        != ""
    ]

    # Remove duplicates
    chunk = chunk.drop_duplicates(
        subset=[
            "comment_id",
            "entry_id",
            "user",
            "comment_clean"
        ]
    )

    # Detect English
    english_flags = (
        detect_english_parallel(
            chunk[
                "comment_clean"
            ].tolist()
        )
    )

    # Keep English only
    chunk = chunk[
        english_flags
    ].copy()

    total_kept += len(chunk)

    # Save
    chunk.to_csv(
        output_file,
        mode=(
            "w"
            if first_chunk
            else "a"
        ),
        header=first_chunk,
        index=False
    )

    first_chunk = False

    print(
        f"Chunk {chunk_number} completed | "
        f"Processed: {total_rows:,} | "
        f"English kept: {total_kept:,}"
    )

print("\nPreprocessing complete.")

print(
    "Total raw rows processed:",
    f"{total_rows:,}"
)

print(
    "Total English comments kept:",
    f"{total_kept:,}"
)

Chunk 1 completed | Processed: 10,000 | English kept: 3,073
Chunk 2 completed | Processed: 20,000 | English kept: 5,931
Chunk 3 completed | Processed: 30,000 | English kept: 8,567
Chunk 4 completed | Processed: 40,000 | English kept: 13,956
Chunk 5 completed | Processed: 50,000 | English kept: 17,351
Chunk 6 completed | Processed: 60,000 | English kept: 19,751
Chunk 7 completed | Processed: 70,000 | English kept: 22,479
Chunk 8 completed | Processed: 80,000 | English kept: 24,967
Chunk 9 completed | Processed: 90,000 | English kept: 26,975
Chunk 10 completed | Processed: 100,000 | English kept: 30,460
Chunk 11 completed | Processed: 110,000 | English kept: 35,314
Chunk 12 completed | Processed: 120,000 | English kept: 37,605
Chunk 13 completed | Processed: 130,000 | English kept: 40,251
Chunk 14 completed | Processed: 140,000 | English kept: 42,152
Chunk 15 completed | Processed: 150,000 | English kept: 44,307
Chunk 16 completed | Processed: 160,000 | English kept: 46,494
Chunk 17 comp

In [43]:
clean_df = pd.read_csv(
    output_file
)

print(
    "Shape:",
    clean_df.shape
)

clean_df.head()

Shape: (110336, 8)


,comment_id,entry_id,user,source,url,timestamp,comment,comment_clean
0,e/624ca9226b6526ebdb69f9b46df482c7/c/32c6bf5bc...,e/624ca9226b6526ebdb69f9b46df482c7,guardianuk,NaN,NaN,2010-08-06 14:45:07,Reel Review video: Catherine Shoard defends Kn...,Reel Review video: Catherine Shoard defends Kn...
1,e/967b4db48fa74021b24ccbc93c55a61c/c/57905e983...,e/967b4db48fa74021b24ccbc93c55a61c,massoptimization,Bookmarklet,http://friendfeed.com/share/bookmarklet,2010-08-06 15:06:44,Article by at 2010-08-06 09:42:14 Categoriz...,Article by at 2010-08-06 09:42:14 Categorized ...
2,e/79e585effdf640e988539e2dd68c2c6d/c/853946596...,e/79e585effdf640e988539e2dd68c2c6d,sheribabysph,Blip.fm,http://blip.fm/,2010-08-06 15:06:44,Happy Friday! @Skyblue101 and @TropicsZ4: Matc...,Happy Friday! @Skyblue101 and @TropicsZ4: Matc...
3,e/01e9a149fff85ff948972896145d3d65/c/e87986e73...,e/01e9a149fff85ff948972896145d3d65,inhabitat24,NaN,NaN,2010-08-06 15:06:27,Yesterday the enviously green city of Portland...,Yesterday the enviously green city of Portland...
4,e/01e9a149fff85ff948972896145d3d65/c/c70184cfd...,e/01e9a149fff85ff948972896145d3d65,inhabitat24,NaN,NaN,2010-08-06 15:06:27,Yesterday the enviously green city of Portland...,Yesterday the enviously green city of Portland...


In [44]:
print(
    "Missing cleaned comments:",
    clean_df[
        "comment_clean"
    ].isna().sum()
)

print(
    "Duplicate comments:",
    clean_df.duplicated(
        subset=[
            "comment_id",
            "entry_id",
            "user",
            "comment_clean"
        ]
    ).sum()
)

print(
    "Unique entries:",
    clean_df[
        "entry_id"
    ].nunique()
)

Missing cleaned comments: 0
Duplicate comments: 0
Unique entries: 91071


In [45]:
comment_dates = pd.to_datetime(
    clean_df["timestamp"],
    errors="coerce"
)

print(
    "Earliest comment:",
    comment_dates.min()
)

print(
    "Latest comment:",
    comment_dates.max()
)

Earliest comment: 2008-06-20 21:04:39
Latest comment: 2010-08-12 12:09:20
